# 05 Result Analysis

모델별 성능과 주요 변수 중요도를 보고서용 표로 정리합니다.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

classification = pd.read_csv(ROOT / 'outputs/metrics/classification_metrics.csv', encoding='utf-8-sig')
regression = pd.read_csv(ROOT / 'outputs/metrics/regression_metrics.csv', encoding='utf-8-sig')

In [ ]:
classification.pivot_table(index='experiment', columns='model', values='f1')

In [ ]:
regression.pivot_table(index='experiment', columns='model', values='rmse')

In [ ]:
importance_path = ROOT / 'outputs/metrics/feature_importance/classification_all_random_forest_is_success_3000000.csv'
importance = pd.read_csv(importance_path, encoding='utf-8-sig')
importance.head(20)

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'src'))
from preprocessing import get_feature_columns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

features = pd.read_csv(ROOT / 'data/processed/movie_features.csv', encoding='utf-8-sig')
feature_cols = get_feature_columns(features)
matrix = features[feature_cols].apply(pd.to_numeric, errors='coerce')
scaled = StandardScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(matrix))
pca_values = PCA(n_components=2).fit_transform(scaled)
features['pca_1'] = pca_values[:, 0]
features['pca_2'] = pca_values[:, 1]
features['cluster'] = KMeans(n_clusters=4, random_state=42, n_init='auto').fit_predict(scaled)
features.groupby('cluster')['audience_count'].agg(['count', 'mean', 'median']).sort_values('mean', ascending=False)